# 1. Cek GPU dan Environment

## 1.1 Cek Spesifikasi GPU dan Device

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory GPU tersedia: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)} GB")

# 2. Instalasi Library dan Import Utama

## 2.1 Install Library yang Dibutuhkan

In [ ]:
# Install library utama
!pip install torch torchvision torchaudio --upgrade --quiet
!pip install albumentations==1.3.0 --quiet
!pip install segmentation-models-pytorch==0.3.3 --quiet
!pip install opencv-python matplotlib tqdm seaborn scikit-learn timm torchinfo --quiet

## 2.2 Import Library Utama

In [ ]:
import json
import os
import random
import sys
import time

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm import tqdm

# 3. Load Dataset CWFID

## 3.1 Download dan Ekstrak Dataset CWFID

In [ ]:
# Download dan ekstrak dataset CWFID jika belum tersedia
pardir = "dataset-1.0"

if not os.path.exists(pardir):
    !wget -q https://github.com/cwfid/dataset/archive/v1.0.tar.gz
    !tar -zxf v1.0.tar.gz

print("Isi direktori kerja:", os.listdir())
print("Images Folder (5 pertama):", sorted(os.listdir(os.path.join(pardir, "images")))[:5])
print("Masks Folder (5 pertama):", sorted(os.listdir(os.path.join(pardir, "annotations")))[:5])

## 3.2 Muat Daftar Path Gambar dan Mask

In [ ]:
# Path folder gambar dan anotasi (default Colab: /content/dataset-1.0)
image_dir = os.path.join(pardir, "images")
mask_dir = os.path.join(pardir, "annotations")

valid_ext = (".png", ".jpg", ".jpeg")

image_paths = sorted([
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.lower().endswith(valid_ext)
])

mask_paths = sorted([
    os.path.join(mask_dir, f)
    for f in os.listdir(mask_dir)
    if f.lower().endswith(valid_ext)
])

assert len(image_paths) == len(mask_paths), "Jumlah gambar dan mask tidak sama!"
print(f"Total Gambar dimuat : {len(image_paths)}")
print(f"Total Mask dimuat   : {len(mask_paths)}")
print(f"Contoh pasangan     : {os.path.basename(image_paths[0])} <-> {os.path.basename(mask_paths[0])}")

## 3.3 Inspeksi Mask dan Fungsi Konversi RGB Mask ke Label

In [ ]:
# Inspeksi bentuk dan nilai piksel mask asli
sample_mask = Image.open(mask_paths[0])
sample_mask_np = np.array(sample_mask)

print("Mask shape:", sample_mask_np.shape)
print("Unique pixel values:", np.unique(sample_mask_np))


def convert_rgb_mask_to_label(mask_rgb):
    """
    Konversi RGB mask (H, W, 3) -> label mask (H, W):
      - Black (0, 0, 0)   -> 0 (Background)
      - Green (0, 255, 0) -> 1 (Crop)
      - Red   (255, 0, 0) -> 2 (Weed)
    """
    label_mask = np.zeros((mask_rgb.shape[0], mask_rgb.shape[1]), dtype=np.uint8)

    background = np.all(mask_rgb == [0, 0, 0], axis=-1)
    crop = np.all(mask_rgb == [0, 255, 0], axis=-1)
    weed = np.all(mask_rgb == [255, 0, 0], axis=-1)

    label_mask[background] = 0
    label_mask[crop] = 1
    label_mask[weed] = 2

    return label_mask

## 3.4 Visualisasi Sampel Gambar dan Ground Truth Mask Asli

In [ ]:
def visualize_image_and_mask(image_path, mask_path, class_colors=None):
    img = Image.open(image_path).convert("RGB")
    mask_rgb = Image.open(mask_path).convert("RGB")
    label_mask = convert_rgb_mask_to_label(np.array(mask_rgb))

    if class_colors is None:
        class_colors = {
            0: [0, 0, 0],      # Background -> Black
            1: [0, 255, 0],    # Crop -> Green
            2: [255, 0, 0],    # Weed -> Red
        }

    color_mask = np.zeros((label_mask.shape[0], label_mask.shape[1], 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        color_mask[label_mask == cls] = color

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(color_mask)
    plt.title("Ground Truth Mask (Colored)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_image_and_mask(image_paths[0], mask_paths[0])

# 4. Split Dataset (Train / Validation / Test)

## 4.1 Fungsi Split Dataset pada Level Gambar

In [ ]:
def split_dataset_paths(
    image_paths,
    mask_paths,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    shuffle=True,
    seed=42,
):
    """
    Membagi pasangan (image_paths, mask_paths) pada level gambar SEBELUM patching
    agar tidak terjadi data leakage antar split.
    """
    assert len(image_paths) == len(mask_paths), "Jumlah image_paths dan mask_paths harus sama"
    assert np.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Total split ratio harus 1.0"

    total_images = len(image_paths)
    indices = np.arange(total_images)

    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    train_end = int(train_ratio * total_images)
    val_end = train_end + int(val_ratio * total_images)

    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    train_images = [image_paths[i] for i in train_idx]
    train_masks = [mask_paths[i] for i in train_idx]

    val_images = [image_paths[i] for i in val_idx]
    val_masks = [mask_paths[i] for i in val_idx]

    test_images = [image_paths[i] for i in test_idx]
    test_masks = [mask_paths[i] for i in test_idx]

    print(f"Total Images : {total_images}")
    print(f"Train Set    : {len(train_images)} images ({train_ratio * 100:.0f}%)")
    print(f"Valid Set    : {len(val_images)} images ({val_ratio * 100:.0f}%)")
    print(f"Test Set     : {len(test_images)} images ({test_ratio * 100:.0f}%)")

    return (train_images, train_masks), (val_images, val_masks), (test_images, test_masks)

## 4.2 Eksekusi Split Dataset (70% Train / 20% Val / 10% Test)

In [ ]:
(train_images, train_masks), (val_images, val_masks), (test_images, test_masks) = split_dataset_paths(
    image_paths=image_paths,
    mask_paths=mask_paths,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    shuffle=True,
    seed=42,
)

# 5. Patching Dataset

## 5.1 Definisi Class CWFIDPatchDataset (Sliding Window Patching)

In [ ]:
class CWFIDPatchDataset(Dataset):
    """
    Dataset on-the-fly sliding window patching dari daftar path gambar dan mask
    yang sudah dipisahkan per split (train/val/test).
    """
    def __init__(self, image_paths, mask_paths, patch_size=512, stride=256, transform=None):
        assert len(image_paths) == len(mask_paths), "Jumlah image_paths dan mask_paths harus sama"
        self.image_paths = list(image_paths)
        self.mask_paths = list(mask_paths)
        self.patch_size = patch_size
        self.stride = stride
        self.transform = transform

        self.patches = self.create_patch_index()

    def create_patch_index(self):
        patch_list = []
        for idx, img_path in enumerate(self.image_paths):
            with Image.open(img_path) as img:
                w, h = img.size
            for y in range(0, h - self.patch_size + 1, self.stride):
                for x in range(0, w - self.patch_size + 1, self.stride):
                    patch_list.append((idx, x, y))
        return patch_list

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):
        idx, x, y = self.patches[index]

        img = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("RGB")

        img_patch = img.crop((x, y, x + self.patch_size, y + self.patch_size))
        mask_patch = mask.crop((x, y, x + self.patch_size, y + self.patch_size))

        img_patch_np = np.array(img_patch)
        mask_patch_label = convert_rgb_mask_to_label(np.array(mask_patch))

        if self.transform is not None:
            augmented = self.transform(image=img_patch_np, mask=mask_patch_label)
            img_tensor = augmented["image"]
            mask_tensor = torch.as_tensor(augmented["mask"], dtype=torch.long)
        else:
            img_tensor = torch.from_numpy(img_patch_np).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.as_tensor(mask_patch_label, dtype=torch.long)

        return img_tensor, mask_tensor

## 5.2 Pembuatan Patch untuk Train, Validation, dan Test Set

In [ ]:
PATCH_SIZE = 512
STRIDE = 256

# Patching dilakukan terpisah pada masing-masing split (tanpa augmentasi terlebih dahulu)
train_dataset = CWFIDPatchDataset(
    image_paths=train_images,
    mask_paths=train_masks,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    transform=None,
)

val_dataset = CWFIDPatchDataset(
    image_paths=val_images,
    mask_paths=val_masks,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    transform=None,
)

test_dataset = CWFIDPatchDataset(
    image_paths=test_images,
    mask_paths=test_masks,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    transform=None,
)

total_patches = len(train_dataset) + len(val_dataset) + len(test_dataset)
print(f"Total Patches : {total_patches}")
print(f"Train Patches : {len(train_dataset)}")
print(f"Val Patches   : {len(val_dataset)}")
print(f"Test Patches  : {len(test_dataset)}")

# 6. Augmentasi Dataset dan DataLoader

## 6.1 Definisi Pipeline Augmentasi (Train) dan Transformasi Evaluasi (Val/Test)

In [ ]:
# Augmentasi + normalisasi khusus untuk data training
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.RandomCrop(512, 512, p=1.0),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2(transpose_mask=True),
])

# Normalisasi saja (tanpa augmentasi acak) untuk data validation dan test
eval_transform = A.Compose([
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2(transpose_mask=True),
])

## 6.2 Penerapan Transformasi pada Dataset

In [ ]:
train_dataset.transform = train_transform
val_dataset.transform = eval_transform
test_dataset.transform = eval_transform

print("Transformasi berhasil diterapkan: train_transform -> train_dataset | eval_transform -> val & test dataset")

## 6.3 Visualisasi Sampel Patch Setelah Augmentasi

In [ ]:
def visualize_dataset_sample(dataset, index=0, class_colors=None):
    img_patch, mask_patch = dataset[index]

    # Convert Tensor ke numpy array dan unnormalize
    img_np = img_patch.permute(1, 2, 0).cpu().numpy()
    img_np = np.clip(img_np * 0.5 + 0.5, 0, 1)

    mask_np = mask_patch.cpu().numpy()
    if mask_np.ndim == 3 and mask_np.shape[2] == 3:
        mask_np = convert_rgb_mask_to_label(mask_np)

    if class_colors is None:
        class_colors = {
            0: [0, 0, 0],      # Background -> Black
            1: [0, 255, 0],    # Crop -> Green
            2: [255, 0, 0],    # Weed -> Red
        }

    color_mask = np.zeros((mask_np.shape[0], mask_np.shape[1], 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        color_mask[mask_np == cls] = color

    alpha = 0.4
    overlay = np.clip((1 - alpha) * img_np + alpha * (color_mask / 255.0), 0, 1)

    plt.figure(figsize=(24, 6))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.title(f"Patch Image #{index}")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap="gray")
    plt.title(f"Label Mask #{index}")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(color_mask)
    plt.title(f"Colored Mask #{index}")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.title(f"Overlay Patch + Mask #{index}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_dataset_sample(train_dataset, index=1)

## 6.4 Pembuatan DataLoader (Train, Validation, dan Test)

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Batch count -> Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# 7. Setup Student Model (Fast-SCNN From Scratch)

## 7.1 Clone Repository Fast-SCNN PyTorch

In [ ]:
fast_scnn_repo = "/content/Fast-SCNN-pytorch"

if not os.path.exists(fast_scnn_repo):
    !git clone https://github.com/Tramac/Fast-SCNN-pytorch.git /content/Fast-SCNN-pytorch

print("Isi repository Fast-SCNN-pytorch:", os.listdir(fast_scnn_repo))

## 7.2 Inisialisasi Arsitektur Fast-SCNN

In [ ]:
if "/content/Fast-SCNN-pytorch" not in sys.path:
    sys.path.append("/content/Fast-SCNN-pytorch")

from models.fast_scnn import FastSCNN

student_model = FastSCNN(num_classes=3).to(device)
print(f"Student Model Loaded: Fast-SCNN (From Scratch) on {device}")

## 7.3 Ringkasan Arsitektur Model (Torchinfo Summary)

In [ ]:
summary(student_model, input_size=(1, 3, 512, 512))

# 8. Loss Function, Optimizer, Scheduler, dan Early Stopping

## 8.1 Definisi Dice Loss dan Focal Loss

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets, num_classes=3):
        """
        inputs : logits [B, C, H, W]
        targets: labels [B, H, W]
        """
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        inputs_soft = F.softmax(inputs, dim=1)

        dims = (0, 2, 3)
        intersection = torch.sum(inputs_soft * targets_one_hot, dims)
        cardinality = torch.sum(inputs_soft + targets_one_hot, dims)

        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_score.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction="mean"):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce_loss = nn.CrossEntropyLoss(reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce_loss(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss

## 8.2 Definisi Combined Student Loss (Dice + Focal Loss)

In [ ]:
focal_loss_fn = FocalLoss(alpha=1, gamma=2, reduction="mean")
dice_loss_fn = DiceLoss(smooth=1e-5)


def student_loss(student_outputs, labels, dice_weight=0.5, focal_weight=0.5):
    """Gabungan Dice Loss dan Focal Loss untuk training Fast-SCNN from scratch."""
    focal_loss = focal_loss_fn(student_outputs, labels)
    dice_loss = dice_loss_fn(student_outputs, labels, num_classes=3)
    return dice_weight * dice_loss + focal_weight * focal_loss

## 8.3 Inisialisasi Optimizer, Scheduler, dan Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False

    def __call__(self, val_iou):
        if val_iou - self.best_score > self.min_delta:
            self.best_score = val_iou
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-4, weight_decay=1e-5)
print(f"Initial Learning Rate: {optimizer.param_groups[0]['lr']}")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",        # maximize val_iou
    factor=0.5,        # turunkan LR menjadi setengah
    patience=25,       # tunggu 25 epoch tanpa peningkatan val_iou
    threshold=1e-4,    # toleransi perubahan minimum
    min_lr=1e-6,       # batas bawah LR
)

## 8.4 Fungsi Perhitungan Metrik dan Validasi Satu Epoch

In [ ]:
def calculate_metrics(outputs, masks, num_classes=3):
    """
    outputs: logits dari model, shape (B, C, H, W)
    masks  : ground truth label, shape (B, H, W)
    """
    preds = torch.argmax(outputs, dim=1).cpu().numpy()
    masks = masks.cpu().numpy()

    iou_list = []
    dice_list = []

    for cls in range(num_classes):
        tp = np.logical_and(preds == cls, masks == cls).sum()
        fp = np.logical_and(preds == cls, masks != cls).sum()
        fn = np.logical_and(preds != cls, masks == cls).sum()

        iou_list.append(tp / (tp + fp + fn + 1e-7))
        dice_list.append(2 * tp / (2 * tp + fp + fn + 1e-7))

    pixel_acc = (preds == masks).sum() / np.prod(masks.shape)
    return float(np.mean(iou_list)), float(np.mean(dice_list)), float(pixel_acc)


def validate_epoch(loader, model):
    model.eval()
    total_iou, total_dice, total_acc = 0.0, 0.0, 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs_raw = model(images)
            outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw

            iou, dice, acc = calculate_metrics(outputs, masks)
            total_iou += iou
            total_dice += dice
            total_acc += acc

    num_batches = len(loader)
    return total_iou / num_batches, total_dice / num_batches, total_acc / num_batches

# 9. Training Loop Student Model (Fast-SCNN From Scratch)

## 9.1 Eksekusi Training Loop dan Penyimpanan Best Checkpoint

In [ ]:
history = {
    "train_loss": [],
    "val_iou": [],
    "val_dice": [],
    "val_acc": [],
    "lr": [],
}

epochs = 500
best_val_iou = 0.0
best_val_dice = 0.0
early_stopper = EarlyStopping(patience=25, min_delta=1e-4)

for epoch in range(epochs):
    print(f"\nEpoch [{epoch + 1}/{epochs}]")

    student_model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc="Training", leave=False)
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        student_raw = student_model(images)
        student_outputs = student_raw[0] if isinstance(student_raw, (tuple, list)) else student_raw

        loss = student_loss(student_outputs, labels, dice_weight=0.5, focal_weight=0.5)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    val_iou, val_dice, val_acc = validate_epoch(val_loader, student_model)

    scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Learning Rate: {current_lr:.8f}")

    history["train_loss"].append(avg_train_loss)
    history["val_iou"].append(val_iou)
    history["val_dice"].append(val_dice)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    print(
        f"Train Loss: {avg_train_loss:.4f} | Val IoU: {val_iou:.4f} | "
        f"Val Dice: {val_dice:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(student_model.state_dict(), "best_scracth_student_model_iou.pth")
        print("Best IoU Model Saved!")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(student_model.state_dict(), "best_scracth_student_model_dice.pth")
        print("Best Dice Model Saved!")

    early_stopper(val_iou)
    if early_stopper.early_stop:
        print("Early stopping triggered!")
        break

## 9.2 Visualisasi Grafik Training History (Loss, IoU, Dice, dan Accuracy)

In [ ]:
def plot_student_history(history):
    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history["train_loss"], "b-", label="Train Loss")
    plt.title("Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history["val_iou"], "g-", label="Val IoU")
    plt.plot(epochs_range, history["val_dice"], "r-", label="Val Dice")
    plt.plot(epochs_range, history["val_acc"], "b-", label="Val Acc")
    plt.title("Validation Metrics")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_student_history(history)

## 9.3 Visualisasi Perubahan Learning Rate per Epoch

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["lr"], marker="o")
plt.title("Learning Rate Progress During Training")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.grid(True)
plt.show()

## 9.4 Simpan Training History (JSON & CSV) ke Google Drive

In [ ]:
save_path = "/content/drive/MyDrive/colab_indra/ss/cwfid_new/fast_scnn"
os.makedirs(save_path, exist_ok=True)

history_filename_json = os.path.join(save_path, "student_training_history.json")
with open(history_filename_json, "w") as f:
    json.dump(history, f)
print(f"History JSON saved to: {history_filename_json}")

df_history = pd.DataFrame(history)
history_filename_csv = os.path.join(save_path, "student_history.csv")
df_history.to_csv(history_filename_csv, index=False)
print(f"History CSV saved to: {history_filename_csv}")

## 9.5 Simpan Model Fast-SCNN Terbaik ke Google Drive

In [ ]:
save_path = "/content/drive/MyDrive/colab_indra/ss/cwfid_new/fast_scnn"
os.makedirs(save_path, exist_ok=True)

best_model_path = os.path.join(save_path, "best_fast_scnn_student_model.pth")
torch.save(student_model.state_dict(), best_model_path)
print(f"Model Fast-SCNN Terbaik Disimpan di: {best_model_path}")

# 10. Evaluasi Model

## 10.1 Fungsi Evaluasi Metrik Macro dan Confusion Matrix

In [ ]:
def evaluate_metrics(model, loader, dataset_name="Dataset", num_classes=3, class_names=None):
    model.eval()
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs_raw = model(imgs)
            outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())

    all_preds = np.concatenate(all_preds).flatten()
    all_masks = np.concatenate(all_masks).flatten()

    precision = precision_score(all_masks, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_masks, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_masks, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_masks, all_preds, labels=list(range(num_classes)))

    print(f"=== {dataset_name} Metrics ===")
    print(f"Precision (macro): {precision:.4f}")
    print(f"Recall (macro):    {recall:.4f}")
    print(f"F1-Score (macro):  {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    if class_names:
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
        )
        plt.title(f"{dataset_name} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.show()

    return precision, recall, f1, cm

## 10.2 Eksekusi Evaluasi Macro pada Train, Validation, dan Test Set

In [ ]:
student_model.to(device)
class_names = ["Background", "Crop", "Weed"]

train_precision, train_recall, train_f1, train_cm = evaluate_metrics(
    student_model, train_loader, dataset_name="Training", class_names=class_names
)
val_precision, val_recall, val_f1, val_cm = evaluate_metrics(
    student_model, val_loader, dataset_name="Validation", class_names=class_names
)
test_precision, test_recall, test_f1, test_cm = evaluate_metrics(
    student_model, test_loader, dataset_name="Test", class_names=class_names
)

## 10.3 Fungsi Evaluasi Metrik Per Kelas

In [ ]:
def evaluate_per_class_metrics(model, loader, class_names=None, num_classes=3):
    model.eval()
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs_raw = model(imgs)
            outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu().numpy().flatten())
            all_masks.append(masks.cpu().numpy().flatten())

    all_preds = np.concatenate(all_preds)
    all_masks = np.concatenate(all_masks)

    cm = confusion_matrix(all_masks, all_preds, labels=list(range(num_classes)))
    cm_percent = np.round(cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-7) * 100, 2)

    precision = precision_score(all_masks, all_preds, labels=list(range(num_classes)), average=None, zero_division=0)
    recall = recall_score(all_masks, all_preds, labels=list(range(num_classes)), average=None, zero_division=0)
    f1 = f1_score(all_masks, all_preds, labels=list(range(num_classes)), average=None, zero_division=0)

    iou_per_class = []
    for cls in range(num_classes):
        tp = cm[cls, cls]
        fp = cm[:, cls].sum() - tp
        fn = cm[cls, :].sum() - tp
        iou_per_class.append(tp / (tp + fp + fn + 1e-7))

    print("Class-wise Metrics:\n")
    for cls in range(num_classes):
        cname = class_names[cls] if class_names else f"Class {cls}"
        print(f"Class: {cname}")
        print(f"  Precision: {precision[cls]:.4f}")
        print(f"  Recall   : {recall[cls]:.4f}")
        print(f"  F1-Score : {f1[cls]:.4f}")
        print(f"  IoU      : {iou_per_class[cls]:.4f}\n")

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_percent, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=class_names if class_names else range(num_classes),
        yticklabels=class_names if class_names else range(num_classes),
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix (%) per Class")
    plt.show()

    return precision, recall, f1, iou_per_class, cm_percent

## 10.4 Eksekusi Evaluasi Per Kelas pada Train, Validation, dan Test Set

In [ ]:
class_names = ["Background", "Crop", "Weed"]
evaluate_per_class_metrics(student_model, train_loader, class_names=class_names)
evaluate_per_class_metrics(student_model, val_loader, class_names=class_names)
evaluate_per_class_metrics(student_model, test_loader, class_names=class_names)

## 10.5 Fungsi Evaluasi Head-to-Head Style McCool

In [ ]:
def evaluate_mccool_metrics(pred_mask, gt_mask, num_classes=3, class_names=None):
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    cm = confusion_matrix(gt_flat, pred_flat, labels=list(range(num_classes)))
    overall_acc = np.sum(np.diag(cm)) / np.sum(cm)

    precision = precision_score(gt_flat, pred_flat, labels=list(range(num_classes)), average=None, zero_division=0)
    recall = recall_score(gt_flat, pred_flat, labels=list(range(num_classes)), average=None, zero_division=0)

    fpr, fnr = [], []
    for cls in range(num_classes):
        tp = cm[cls, cls]
        fn = cm[cls, :].sum() - tp
        fp = cm[:, cls].sum() - tp
        tn = np.sum(cm) - (tp + fp + fn)

        fpr.append(fp / (fp + tn + 1e-7))
        fnr.append(fn / (fn + tp + 1e-7))

    print("======================== McCool Style Metrics ========================")
    print(f"Overall Pixel-wise Accuracy (ACC): {overall_acc:.4f}")
    for cls in range(num_classes):
        name = class_names[cls] if class_names else f"Class {cls}"
        print(f"\nClass: {name}")
        print(f"  Precision: {precision[cls]:.4f}")
        print(f"  Recall   : {recall[cls]:.4f}")
        print(f"  FPR      : {fpr[cls] * 100:.2f}%")
        print(f"  FNR      : {fnr[cls] * 100:.2f}%")

    return overall_acc, precision, recall, fpr, fnr


def evaluate_dataset_mccool(model, loader, dataset_name="Dataset", class_names=None, num_classes=3):
    model.eval()
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                outputs_raw = model(imgs)
                outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu().numpy().flatten())
            all_masks.append(masks.cpu().numpy().flatten())

    all_preds = np.concatenate(all_preds)
    all_masks = np.concatenate(all_masks)

    print(f"\n====================== {dataset_name} ======================")
    return evaluate_mccool_metrics(all_preds, all_masks, num_classes=num_classes, class_names=class_names)

## 10.6 Eksekusi Evaluasi Style McCool pada Train, Validation, dan Test Set

In [ ]:
class_names = ["Background", "Crop", "Weed"]

evaluate_dataset_mccool(student_model, train_loader, dataset_name="TRAIN", class_names=class_names)
evaluate_dataset_mccool(student_model, val_loader, dataset_name="VALIDATION", class_names=class_names)
evaluate_dataset_mccool(student_model, test_loader, dataset_name="TEST", class_names=class_names)

# 11. Inferensi dan Visualisasi Prediksi

## 11.1 Fungsi Sliding Window Inference (Full Image dan Array)

In [ ]:
def infer_full_image(image_path, model, patch_size=512, stride=256, num_classes=3):
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img)

    # Normalisasi sesuai preprocessing training (tanpa augmentasi)
    img_norm = (img_np / 255.0 - 0.5) / 0.5
    img_norm = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    _, _, H, W = img_norm.shape
    pred_mask = torch.zeros((num_classes, H, W), device=device)
    count_mask = torch.zeros((1, H, W), device=device)

    y_steps = list(range(0, H - patch_size + 1, stride))
    x_steps = list(range(0, W - patch_size + 1, stride))

    if y_steps[-1] + patch_size < H:
        y_steps.append(H - patch_size)
    if x_steps[-1] + patch_size < W:
        x_steps.append(W - patch_size)

    with torch.no_grad():
        for y in y_steps:
            for x in x_steps:
                patch = img_norm[:, :, y:y + patch_size, x:x + patch_size]
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    output_raw = model(patch)
                    output = output_raw[0] if isinstance(output_raw, (tuple, list)) else output_raw

                pred_mask[:, y:y + patch_size, x:x + patch_size] += output.squeeze(0)
                count_mask[:, y:y + patch_size, x:x + patch_size] += 1

    pred_mask /= count_mask
    final_pred = torch.argmax(pred_mask, dim=0).cpu().numpy()
    return img_np, final_pred


def infer_from_array(image_array, model, patch_size=512, stride=256, num_classes=3):
    model.eval()

    img_norm = (image_array / 255.0 - 0.5) / 0.5
    img_norm = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    _, _, H, W = img_norm.shape
    pred_mask = torch.zeros((num_classes, H, W), device=device)
    count_mask = torch.zeros((1, H, W), device=device)

    y_steps = list(range(0, H - patch_size + 1, stride))
    x_steps = list(range(0, W - patch_size + 1, stride))

    if y_steps[-1] + patch_size < H:
        y_steps.append(H - patch_size)
    if x_steps[-1] + patch_size < W:
        x_steps.append(W - patch_size)

    with torch.no_grad():
        for y in y_steps:
            for x in x_steps:
                patch = img_norm[:, :, y:y + patch_size, x:x + patch_size]
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    output_raw = model(patch)
                    output = output_raw[0] if isinstance(output_raw, (tuple, list)) else output_raw

                pred_mask[:, y:y + patch_size, x:x + patch_size] += output.squeeze(0)
                count_mask[:, y:y + patch_size, x:x + patch_size] += 1

    pred_mask /= count_mask
    final_pred = torch.argmax(pred_mask, dim=0).cpu().numpy()
    return final_pred

## 11.2 Fungsi Evaluasi dan Visualisasi Prediksi Single Image

In [ ]:
def evaluate_single_image(pred_mask, gt_mask, num_classes=3, class_names=None):
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    cm = confusion_matrix(gt_flat, pred_flat, labels=list(range(num_classes)))
    precision = precision_score(gt_flat, pred_flat, labels=list(range(num_classes)), average=None, zero_division=0)
    recall = recall_score(gt_flat, pred_flat, labels=list(range(num_classes)), average=None, zero_division=0)
    f1 = f1_score(gt_flat, pred_flat, labels=list(range(num_classes)), average=None, zero_division=0)

    iou_per_class = []
    for cls in range(num_classes):
        tp = cm[cls, cls]
        fp = cm[:, cls].sum() - tp
        fn = cm[cls, :].sum() - tp
        iou_per_class.append(tp / (tp + fp + fn + 1e-7))

    print("Per-Class Evaluation Metrics:")
    for cls in range(num_classes):
        name = class_names[cls] if class_names else f"Class {cls}"
        print(
            f"  {name:10s} | Precision: {precision[cls]:.4f}, "
            f"Recall: {recall[cls]:.4f}, F1: {f1[cls]:.4f}, IoU: {iou_per_class[cls]:.4f}"
        )

    return precision, recall, f1, iou_per_class, cm


def visualize_full_prediction(img_np, pred_mask, gt_mask=None, class_colors=None, alpha=0.4, class_names=None):
    H, W, _ = img_np.shape

    if class_colors is None:
        class_colors = {
            0: [0, 0, 0],      # Background -> Black
            1: [0, 255, 0],    # Crop -> Green
            2: [255, 0, 0],    # Weed -> Red
        }

    color_pred_mask = np.zeros((H, W, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        color_pred_mask[pred_mask == cls] = color

    overlay_pred = np.clip((1 - alpha) * img_np / 255.0 + alpha * color_pred_mask / 255.0, 0, 1)

    color_gt_mask = np.zeros((H, W, 3), dtype=np.uint8)
    if gt_mask is not None:
        for cls, color in class_colors.items():
            color_gt_mask[gt_mask == cls] = color

    plt.figure(figsize=(24, 6))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(color_gt_mask)
    plt.title("Ground Truth Mask")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(color_pred_mask)
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay_pred)
    plt.title("Overlay Prediction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    if gt_mask is not None:
        return evaluate_single_image(
            pred_mask, gt_mask, num_classes=len(class_colors), class_names=class_names
        )

    print("Ground truth mask not provided, skipping metrics evaluation.")
    return None

## 11.3 Inferensi dan Visualisasi pada Satu Full Image

In [ ]:
sample_image_path = image_paths[0]
sample_gt_mask_path = mask_paths[0]

gt_mask_rgb = Image.open(sample_gt_mask_path).convert("RGB")
gt_label_mask = convert_rgb_mask_to_label(np.array(gt_mask_rgb))

img_np, pred_mask = infer_full_image(
    image_path=sample_image_path,
    model=student_model,
    patch_size=512,
    stride=256,
    num_classes=3,
)

class_names = ["Background", "Crop", "Weed"]
precision, recall, f1, iou, cm = visualize_full_prediction(
    img_np, pred_mask, gt_mask=gt_label_mask, class_names=class_names
)

## 11.4 Inferensi dan Visualisasi pada 5 Sampel Acak Full Image

In [ ]:
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])
random_images = random.sample(image_files, min(5, len(image_files)))
class_names = ["Background", "Crop", "Weed"]

for idx, img_file in enumerate(random_images):
    print(f"\nSample {idx + 1}: {img_file}")

    curr_image_path = os.path.join(image_dir, img_file)
    curr_mask_path = os.path.join(mask_dir, img_file.replace("image", "annotation"))

    gt_mask_rgb = Image.open(curr_mask_path).convert("RGB")
    gt_label_mask = convert_rgb_mask_to_label(np.array(gt_mask_rgb))

    img_np, pred_mask = infer_full_image(
        image_path=curr_image_path,
        model=student_model,
        patch_size=512,
        stride=256,
        num_classes=3,
    )

    visualize_full_prediction(
        img_np, pred_mask, gt_mask=gt_label_mask, class_names=class_names
    )
    print(f"Finished sample {idx + 1}")

## 11.5 Inferensi dan Visualisasi pada 5 Sampel Acak Test Dataset (Patch)

In [ ]:
class_names = ["Background", "Crop", "Weed"]
random_indices = random.sample(range(len(test_dataset)), min(5, len(test_dataset)))

for idx, sample_idx in enumerate(random_indices):
    print(f"\nTest Sample {idx + 1} | Test Dataset index: {sample_idx}")

    image_tensor, gt_label_mask = test_dataset[sample_idx]
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = np.clip((img_np * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)

    pred_mask = infer_from_array(
        image_array=img_np,
        model=student_model,
        patch_size=512,
        stride=256,
        num_classes=3,
    )

    visualize_full_prediction(
        img_np, pred_mask, gt_mask=gt_label_mask.cpu().numpy(), class_names=class_names
    )
    print(f"Finished test sample {idx + 1}")

## 11.6 Inferensi Langsung 512x512 Tanpa Patching (Resize)

In [ ]:
# Random -> Resize 512x512 -> Inference -> Visualisasi
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])
random_images = random.sample(image_files, min(5, len(image_files)))
class_names = ["Background", "Crop", "Weed"]

student_model.eval()

for idx, img_file in enumerate(random_images):
    print(f"\nSample {idx + 1} (Direct 512x512 Resize): {img_file}")

    curr_image_path = os.path.join(image_dir, img_file)
    img = Image.open(curr_image_path).convert("RGB").resize((512, 512))
    img_np = np.array(img)

    img_norm = (img_np / 255.0 - 0.5) / 0.5
    img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    curr_mask_path = os.path.join(mask_dir, img_file.replace("image", "annotation"))
    gt_mask = Image.open(curr_mask_path).convert("RGB").resize((512, 512), resample=Image.NEAREST)
    gt_label_mask = convert_rgb_mask_to_label(np.array(gt_mask))

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            outputs_raw = student_model(img_tensor)
            outputs = outputs_raw[0] if isinstance(outputs_raw, (tuple, list)) else outputs_raw

    pred_mask = torch.argmax(outputs.squeeze(0), dim=0).cpu().numpy()
    img_vis = np.clip((img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)

    visualize_full_prediction(
        img_vis, pred_mask, gt_mask=gt_label_mask, class_names=class_names
    )
    print(f"Finished sample {idx + 1}")

# 12. Pengukuran Kecepatan Inferensi Fast-SCNN (From Scratch) (Latency & FPS)

## 12.1 Fungsi Pengukuran Inference Time dan FPS

In [ ]:
def measure_inference_time(model, input_size=(1, 3, 512, 512), num_runs=100, device="cuda"):
    model.eval()
    dummy_input = torch.randn(input_size).to(device)

    # Warm-up GPU
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.synchronize()
    start_time = time.time()

    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(dummy_input)

    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.synchronize()
    end_time = time.time()

    avg_time_per_run = (end_time - start_time) / num_runs
    print(f"Inference time per image ({input_size[-2]}x{input_size[-1]}): {avg_time_per_run * 1000:.2f} ms")
    print(f"FPS (Frames Per Second): {1.0 / avg_time_per_run:.2f}")

    return avg_time_per_run

## 12.2 Cek Spesifikasi GPU untuk Benchmark

In [ ]:
if torch.cuda.is_available():
    gpu_idx = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(gpu_idx)
    gpu_capability = torch.cuda.get_device_capability(gpu_idx)
    gpu_mem = torch.cuda.get_device_properties(gpu_idx).total_memory / 1024**3

    print(f"GPU Name           : {gpu_name}")
    print(f"Compute Capability : {gpu_capability}")
    print(f"Total VRAM         : {gpu_mem:.2f} GB")
else:
    print("CUDA not available. Using CPU.")

!nvidia-smi

## 12.3 Benchmark Inference Time pada Input Patch 512x512

In [ ]:
student_model.to(device)
measure_inference_time(student_model, input_size=(1, 3, 512, 512), num_runs=100, device=device)

## 12.4 Benchmark Inference Time pada Full Image (Sliding Window)

In [ ]:
benchmark_image_path = sample_image_path

if torch.cuda.is_available() and device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.time()

img_np, pred_mask = infer_full_image(
    image_path=benchmark_image_path,
    model=student_model,
    patch_size=512,
    stride=256,
    num_classes=3,
)

if torch.cuda.is_available() and device.type == "cuda":
    torch.cuda.synchronize()
total_time = time.time() - start_time

print(f"Inference time per full image: {total_time:.3f} sec")
print(f"FPS (full image): {1.0 / total_time:.2f}")